# 01 · Generating looming stimuli

Render a textured object approaching the observer (hit) or passing nearby (miss) with the differentiable PyTorch3D pipeline.

**Requires** `pytorch3d`.

In [ ]:
import torch
from flynet_robustness.textures import get_texture, TEXTURE_TYPES
from flynet_robustness.rendering import generate_looming_stimulus, save_frames_as_gif

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('textures:', TEXTURE_TYPES)

## A single hit trajectory
`generate_looming_stimulus` returns luminance frames `(T, H, W)`, plus speed, angle and 3D positions.

In [ ]:
texture = get_texture('Checkerboard', size=128)
vis, speed, angle, position = generate_looming_stimulus(
    hit=True, radius=4.0, theta=0.0, shape='Sphere', texture=texture,
    time=1.0, fps=100, stop_margin_hit=0.6, device=device, z0=15, out_size=416)
print('frames', tuple(vis.shape), 'speed', round(speed, 3))

In [ ]:
# preview: middle frame
import matplotlib.pyplot as plt
plt.imshow(vis[50].cpu(), cmap='gray', vmin=0, vmax=1); plt.axis('off'); plt.show()

## Angular size along the trajectory

In [ ]:
import numpy as np
from flynet_robustness.rendering import sample_start, compute_trajectory, compute_angular_size_trajectory
start = sample_start(radius=4.0, theta=0.0)
start_pos, dir_vec = compute_trajectory(start, z0=15, z_end=2.0, hit=True)
t, D, ang, ang_speed = compute_angular_size_trajectory(start_pos, dir_vec, R=1.0, T=1.0, sphere=True)
plt.plot(t, np.degrees(ang)); plt.xlabel('time (s)'); plt.ylabel('angular size (deg)'); plt.show()

## Save a GIF

In [ ]:
save_frames_as_gif(vis, path='gifs/hit_sphere.gif', fps=50)